# Is there anything about kidney cancer already?

*Standalone -- no prior vignette required.*

> **DRAFT STATUS:** specific terms ("nephron", "renal cortex"), counts ("a few thousand"), and the exact narrative beats below are placeholders -- this notebook has not been run against the live API. Before finalizing, run each cell for real and replace any invented specifics with actual output. The structure/posture is the reviewed part; the content is not yet verified.

I'm a grad student. I don't have a disease picked out yet, and I don't have much budget to go collect new data myself -- so before I commit to anything, I just want to know: is there *already* data out there about kidney cancer? I don't know what columns CDA has, I don't know what words the data uses internally, and honestly I don't want to find out before I even know if this is worth my time.

So I'm not going to go look anything up first. I'm just going to type the word and see what happens.

In [ ]:
from itables import init_notebook_mode, show
init_notebook_mode(all_interactive=True)
import itables.options as opt
opt.classes="display nowrap compact"
opt.buttons=["copyHtml5", "csvHtml5", "excelHtml5"]
opt.maxBytes=0
opt.columnDefs = [{"className": "dt-left", "targets": "_all"}]

In [ ]:
from cdapython import summarize_subjects
summarize_subjects('kidney')

That's a few thousand subjects, from one word, with zero setup. I didn't tell it which column to look in -- `kidney` could be an anatomic site, a diagnosis, part of a file description, anything -- it just searched everywhere at once and found every subject connected to the term in any way.

I don't know yet if that's useful to me. Let's see what these subjects actually look like before deciding anything. The summary doesn't show every column by default, so let me check what's available and pull in a few that sound relevant:

In [ ]:
from cdapython import columns
columns()

In [ ]:
summarize_subjects('kidney', add_columns=['anatomic_site', 'diagnosis', 'format'])

Interesting -- a lot of these subjects also show up with `vcf` in their file formats. If there's sequencing data sitting around too, that's worth knowing before I pick a direction. I can just add a second word -- I don't need to abandon the first one or restart with a more careful query:

In [ ]:
summarize_subjects('kidney', 'vcf', add_columns=['anatomic_site', 'diagnosis', 'format'])

Two free-text terms like this means "must touch both" -- every subject here is connected to *both* kidney and vcf somewhere in their record, still searched across every column at once. That's still a smaller, more useful number than I expected to get this easily.

One thing I notice scrolling through the `anatomic_site` column: not everything actually says "kidney." Some rows say "nephron." Some say "renal cortex." I only typed one word, so where did those come from?

I didn't do anything to make that happen -- CDA's anatomy fields are backed by a real medical ontology (UBERON), which means every value already knows its own synonyms and its place in a hierarchy of broader/narrower terms. "Kidney" automatically pulls in anything underneath it in that tree -- nephron, renal cortex, renal tubule, all of it -- and anything else doctors or datasets call the same structure. I didn't need to know that ontology existed, let alone look anything up in it, for this to work.

If I'm curious *why* a specific row matched -- was it the literal word, a synonym, or something more specific rolling up -- I can ask CDA to show its work:

In [ ]:
summarize_subjects('kidney', 'vcf', add_columns=['anatomic_site', 'diagnosis', 'format'], add_extras=['all'])

Same subjects, same count -- `add_extras` doesn't change what matched, it just annotates *why*. A row with "nephron" now shows that it matched because nephron rolls up to kidney in the ontology; a row with a straight synonym shows that instead.

I still haven't written a single filter, looked up a controlled vocabulary, or checked what table anything lives in. If this many people have kidney data with sequencing attached, that's enough for me to decide this is worth pursuing -- and if I do, [Getting oriented before I filter](#) is where I'd go next to actually get oriented in what's here before building a real query.